In [1]:
import os

In [2]:
%pwd

'/Users/khanhvu/Documents/Code/MLOPS/mlops-ds-project/research'

In [3]:
os.chdir("../")
%pwd

'/Users/khanhvu/Documents/Code/MLOPS/mlops-ds-project'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [5]:
from src.datascience.config import *
from src.datascience.utils.common import read_yaml, create_directories, load_json, save_json

In [6]:
class ConfiurationManager:
    def __init__(self, config_file_path = CONFIG_FILE_PATH, 
                 params_file_path = PARAMS_FILE_PATH,
                 schema_file_path = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifacts_root,], True)
        
    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )

        return data_ingestion_config


In [22]:
# import requests
import urllib.request as request
from src.datascience import logger
import zipfile
import os

In [24]:

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                self.config.source_URL, self.config.local_data_file
            )
            logger.info(f"File downloaded successfully: {filename}")
        else:
            logger.info(f"File already exists: {self.config.local_data_file}")

    def extract_zip_file(self):


        unzip_dir = self.config.unzip_dir
        if not os.path.exists(unzip_dir):
            os.makedirs(unzip_dir)
            logger.info(f"Directory created: {unzip_dir}")
            
        if os.path.exists(self.config.local_data_file):
            with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
                zip_ref.extractall(self.config.unzip_dir)
            logger.info(f"File extracted successfully to: {self.config.unzip_dir}")
        else:
            logger.error(f"File does not exist: {self.config.local_data_file}")

In [25]:
config = ConfiurationManager()
data_ingestion_config = config.get_data_ingestion_config()
data_ingestion = DataIngestion(config=data_ingestion_config)
data_ingestion.download_file()
data_ingestion.extract_zip_file()

[2026-03-29 16:29:32,577: INFO: common]: created directory at: artifacts
[2026-03-29 16:29:32,578: INFO: common]: created directory at: artifacts/data_ingestion
[2026-03-29 16:29:33,530: INFO: 4122209555]: File downloaded successfully: artifacts/data_ingestion/data.zip
[2026-03-29 16:29:33,533: INFO: 4122209555]: File extracted successfully to: artifacts/data_ingestion
